<a href="https://colab.research.google.com/github/rdelhibabu/VQA_QML/blob/main/VQA_QML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the required quantum computing frameworks and ML tools
!pip install pennylane qiskit qiskit-aer mitiq scikit-learn matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.5/356.5 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 104.1 MB/s eta 0:00:00


In [4]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Import ZNE tools from their new home in the noise module
from pennylane.noise import mitigate_with_zne, fold_global, poly_extrapolate
from functools import partial

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Import ZNE tools from their new home in the noise module
from pennylane.noise import mitigate_with_zne, fold_global, poly_extrapolate
from functools import partial

# ==========================================
# 1. DATA PREPARATION
# ==========================================
np.random.seed(42)
N_SAMPLES = 100
n_qubits = 4

# Encode classical features normalized into the [-pi, pi] range
X = np.random.uniform(-np.pi, np.pi, (N_SAMPLES, n_qubits))

# Binary clinical labels: Baseline (-1) vs. Pre-seizure (1)
y = np.where(np.sin(X[:, 0]) * np.cos(X[:, 1]) > 0, 1, -1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 2. VQC ARCHITECTURE (Defines vqc_circuit)
# ==========================================
layers = 3 # L=3 as specified for the HEA topology
dev = qml.device("default.qubit", wires=n_qubits)

def data_encoding(x):
    """Angle encoding mapping independent classical features into the quantum state."""
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)
    # Static layer of nearest-neighbor CZ gates to capture temporal dynamics
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i+1])

def variational_layer(params):
    """Hardware-efficient layer with single-qubit rotations and linear CNOTs."""
    for i in range(n_qubits):
        qml.RX(params[i, 0], wires=i)
        qml.RY(params[i, 1], wires=i)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])

@qml.qnode(dev)
def vqc_circuit(params, x):
    """Full VQC evaluating the local parity observable to mitigate barren plateaus."""
    data_encoding(x)
    for l in range(layers):
        variational_layer(params[l])

    # Local parity measurement on the first two qubits
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

# ==========================================
# 3. ZERO-NOISE EXTRAPOLATION & TRAINING
# ==========================================
# Define ZNE Noise Scales as per the paper
scale_factors = [1, 3, 5]

# Set up the extrapolation method (Polynomial extrapolation of order 2 is standard for 3 scale factors)
extrapolation_method = partial(poly_extrapolate, order=2)

# Create the mitigated version of the QNode
mitigated_qnode = mitigate_with_zne(
    vqc_circuit,
    scale_factors=scale_factors,
    folding=fold_global,
    extrapolate=extrapolation_method
)

def zne_cost_function(params, X_batch, y_batch):
    """Evaluates the ZNE-mitigated Mean Squared Error (MSE) cost function."""
    total_loss = 0
    for i in range(len(X_batch)):
        x_i = X_batch[i]
        y_true = y_batch[i]

        # The mitigated_qnode automatically executes the folded circuits and extrapolates!
        mitigated_expval = mitigated_qnode(params, x_i)

        # Compute MSE between mitigated expectation and true label
        total_loss += (mitigated_expval - y_true) ** 2

    return total_loss / len(X_batch)

# Optimization hyperparameters
epochs = 50
learning_rate = 0.05
opt = qml.AdamOptimizer(stepsize=learning_rate)

# Applying Identity Initialization: start close to 0 to bypass Haar scrambling variance
params = np.random.uniform(-0.01, 0.01, (layers, n_qubits, 2), requires_grad=True)

print("Starting ZNE-Mitigated Training...")
loss_history = []

for epoch in range(epochs):
    # Optimizing the parameters via Adam
    params, cost = opt.step_and_cost(lambda p: zne_cost_function(p, X_train, y_train), params)
    loss_history.append(cost)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:2d} | MSE Loss: {cost:.4f}")

# Replicate the loss trajectory visualization
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), loss_history, label='QPU Mitigated (ZNE)', color='green', marker='o', markersize=4)
plt.xlabel('Training Epochs')
plt.ylabel('Mean Squared Error (MSE)')
plt.title('Empirical Training Loss Trajectory (Simulated)')
plt.legend()
plt.grid(True)
plt.show()

Starting ZNE-Mitigated Training...
Epoch  5 | MSE Loss: 1.3362
Epoch 10 | MSE Loss: 1.0538
Epoch 15 | MSE Loss: 0.9671
Epoch 20 | MSE Loss: 0.9428
Epoch 25 | MSE Loss: 0.9063
Epoch 30 | MSE Loss: 0.8730
Epoch 35 | MSE Loss: 0.8530
Epoch 40 | MSE Loss: 0.8409
